# Moctale — Movie Recommendation System
A content-based movie recommendation system built with Python, pandas, and scikit-learn. Uses the TMDB 5000 dataset to suggest similar movies based on overview, genres, keywords, cast, and director, ranked by cosine similarity.

In [1]:
import numpy as np
import pandas as pd
import difflib
import ast
import pickle
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [3]:
# Merge on unique movie IDs to prevent duplicate-title cross-multiplication
movies = movies.merge(credits, left_on='id', right_on='movie_id')
movies = movies[['id', 'title_x', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.rename(columns={'id': 'movie_id', 'title_x': 'title'}, inplace=True)
movies.shape

In [4]:
# Handle missing values and reset index to ensure synchronous positional indexing
movies.dropna(subset=['overview'], inplace=True)
movies.reset_index(drop=True, inplace=True)
movies.isnull().sum()

In [5]:
# Helper functions for extracting structured metadata from JSON strings
def convert_genres_keywords(obj):
    L = []
    try:
        for i in ast.literal_eval(obj):
            L.append(i['name'])
    except Exception:
        pass
    return L

def convert_cast(obj):
    L = []
    counter = 0
    try:
        for i in ast.literal_eval(obj):
            if counter != 3:
                L.append(i['name'])
                counter += 1
            else:
                break
    except Exception:
        pass
    return L

def fetch_director(obj):
    L = []
    try:
        for i in ast.literal_eval(obj):
            if i.get('job') == 'Director':
                L.append(i['name'])
                break
    except Exception:
        pass
    return L

In [6]:
movies['genres'] = movies['genres'].apply(convert_genres_keywords)
movies['keywords'] = movies['keywords'].apply(convert_genres_keywords)
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])

In [7]:
# Strip spaces to create atomic entity tags (e.g. 'Sam Worthington' -> 'SamWorthington')
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

# Combine all features into unified tags
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [8]:
new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())
new_df['title_lower'] = new_df['title'].str.lower().str.strip()
new_df.head()

In [9]:
# Apply Porter Stemmer for morphological normalization
ps = PorterStemmer()
def stem(text):
    return " ".join([ps.stem(i) for i in text.split()])

new_df['tags'] = new_df['tags'].apply(stem)

In [10]:
# Vectorize with CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
similarity = cosine_similarity(vectors)
similarity.shape

In [11]:
# Verified recommendation function with fuzzy matching & index safety
def recommend(movie):
    movie = movie.lower().strip()
    matches = new_df[new_df['title_lower'] == movie]
    
    if matches.empty:
        close = difflib.get_close_matches(movie, new_df['title_lower'], n=3, cutoff=0.6)
        if close:
            print(f"'{movie}' not found. Did you mean: {', '.join(close)}?")
        else:
            print(f"'{movie}' not found in the database.")
        return
    
    movie_index = matches.index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    for i in movies_list:
        print(f"{new_df.iloc[i[0]].title} (Score: {i[1]:.4f})")

In [12]:
recommend('Avatar')

In [13]:
# Save serialized model artifacts for deployment
pickle.dump(new_df, open('movie.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print('Model artifacts successfully serialized!')